In [ ]:
# STEP 1: Install dependencies and check hardware
!pip install -q galore-torch transformers datasets sentencepiece

import torch
import transformers
import datasets
import importlib.metadata

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("GaLore:", importlib.metadata.version("galore-torch"))
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(
        f"GPU {i}: {props.name} | "
        f"VRAM: {props.total_memory / 1024**3:.2f} GiB"
    )

if not torch.cuda.is_available():
    raise RuntimeError("Enable GPU acceleration in notebook settings.")

if torch.cuda.device_count() != 2:
    raise RuntimeError("Select GPU T4 ×2 before continuing.")

In [ ]:
# STEP 2: Prepare fixed C4 training and validation data
import os
import time
import torch
from datasets import load_dataset
from transformers import AutoTokenizer

WORK_DIR = "/kaggle/working/galore_reproduction"
os.makedirs(WORK_DIR, exist_ok=True)

SEQ_LEN = 256
TRAIN_BLOCKS = 4096
VAL_BLOCKS = 512

# Tokenizer used in your previous official-code-based setup.
tokenizer = AutoTokenizer.from_pretrained("t5-base")

print("Tokenizer:", tokenizer.name_or_path)
print("Tokenizer size:", len(tokenizer))
print("EOS token ID:", tokenizer.eos_token_id)

def prepare_packed_blocks(split, num_blocks):
    dataset = load_dataset(
        "allenai/c4",
        "en",
        split=split,
        streaming=True,
    )

    blocks = []
    buffer = []
    documents = 0
    start = time.perf_counter()

    for example in dataset:
        text = example["text"]
        if not text.strip():
            continue

        ids = tokenizer(
            text,
            add_special_tokens=False,
            truncation=False,
            verbose=False,
        )["input_ids"]

        # Preserve a boundary between documents.
        ids.append(tokenizer.eos_token_id)
        buffer.extend(ids)
        documents += 1

        available = min(
            len(buffer) // SEQ_LEN,
            num_blocks - len(blocks),
        )

        for j in range(available):
            offset = j * SEQ_LEN
            blocks.append(buffer[offset:offset + SEQ_LEN])

        buffer = buffer[available * SEQ_LEN:]

        if len(blocks) >= num_blocks:
            break

        if documents % 1000 == 0:
            print(
                f"{split}: {len(blocks):,}/{num_blocks:,} blocks "
                f"from {documents:,} documents",
                flush=True,
            )

    if len(blocks) != num_blocks:
        raise RuntimeError(f"Not enough data in {split}.")

    tensor = torch.tensor(blocks, dtype=torch.long)

    print(
        f"{split} ready: shape={tuple(tensor.shape)}, "
        f"tokens={tensor.numel():,}, "
        f"time={time.perf_counter() - start:.1f}s"
    )
    return tensor

train_tokens = prepare_packed_blocks("train", TRAIN_BLOCKS)
val_tokens = prepare_packed_blocks("validation", VAL_BLOCKS)

# Fixed ordering shared by both optimizer experiments.
generator = torch.Generator().manual_seed(42)
order = torch.randperm(len(train_tokens), generator=generator)
train_tokens = train_tokens[order]

assert train_tokens.min().item() >= 0
assert max(
    train_tokens.max().item(),
    val_tokens.max().item(),
) < 32000, "Token IDs exceed the intended model vocabulary."

data_path = os.path.join(WORK_DIR, "benchmark_data.pt")

torch.save(
    {
        "train": train_tokens,
        "validation": val_tokens,
        "sequence_length": SEQ_LEN,
        "tokenizer": tokenizer.name_or_path,
        "seed": 42,
    },
    data_path,
)

print("\nSaved:", data_path)
print("Training tokens:", train_tokens.numel())
print("Validation tokens:", val_tokens.numel())
print("Maximum training token ID:", train_tokens.max().item())

In [ ]:
# STEP 3: Benchmark AdamW and GaLore on separate T4 GPUs
import os
import sys
import json
import subprocess
from pathlib import Path

WORK_DIR = Path("/kaggle/working/galore_reproduction")

benchmark_code = r'''
import os
import sys
import json
import time
import torch
from transformers import LlamaConfig, LlamaForCausalLM
from galore_torch import GaLoreAdamW

method = sys.argv[1]
work_dir = sys.argv[2]

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.set_num_threads(4)

device = torch.device("cuda:0")

config = LlamaConfig(
    vocab_size=32000,
    hidden_size=512,
    intermediate_size=1376,
    num_hidden_layers=8,
    num_attention_heads=8,
    num_key_value_heads=8,
    max_position_embeddings=256,
    hidden_act="silu",
    rms_norm_eps=1e-6,
    bos_token_id=0,
    eos_token_id=1,
    use_cache=False,
)

model = LlamaForCausalLM(config).to(device).float()
model.train()

data = torch.load(
    os.path.join(work_dir, "benchmark_data.pt"),
    weights_only=True,
)
train = data["train"]

# Project attention and feed-forward linear weights.
# Embeddings, output head, and normalization remain unprojected.
projected = []
ordinary = []
projected_names = []

for name, parameter in model.named_parameters():
    if not parameter.requires_grad:
        continue

    if (
        parameter.ndim == 2
        and (
            ".self_attn." in name
            or ".mlp." in name
        )
    ):
        projected.append(parameter)
        projected_names.append(name)
    else:
        ordinary.append(parameter)

common = dict(
    lr=1e-4,
    betas=(0.9, 0.999),
    eps=1e-8,
    weight_decay=0.01,
)

if method == "adamw":
    optimizer = torch.optim.AdamW(
        model.parameters(), **common
    )
else:
    optimizer = GaLoreAdamW(
        [
            {"params": ordinary},
            {
                "params": projected,
                "rank": 128,
                "update_proj_gap": 200,
                "scale": 0.25,
                "proj_type": "std",
            },
        ],
        **common,
    )

batch_size = 16
steps = 220
step_times = []
losses = []

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()
start = time.perf_counter()

for step in range(steps):
    offset = (step * batch_size) % len(train)
    x = train[offset:offset + batch_size].to(device)

    torch.cuda.synchronize()
    step_start = time.perf_counter()

    optimizer.zero_grad(set_to_none=True)
    output = model(input_ids=x, labels=x)
    loss = output.loss

    if not torch.isfinite(loss):
        raise RuntimeError(
            f"{method}: nonfinite loss at step {step}"
        )

    loss.backward()
    optimizer.step()

    torch.cuda.synchronize()
    step_times.append(time.perf_counter() - step_start)
    losses.append(loss.item())

    if (step + 1) % 50 == 0:
        print(
            f"{method}: {step + 1}/{steps}, "
            f"loss={loss.item():.4f}",
            flush=True,
        )

torch.cuda.synchronize()
elapsed = time.perf_counter() - start

# Include tensors stored inside GaLore projector objects.
seen_objects = set()
seen_storages = set()

def tensor_storage_bytes(obj):
    if id(obj) in seen_objects:
        return 0
    seen_objects.add(id(obj))

    if torch.is_tensor(obj):
        storage = obj.untyped_storage()
        key = (str(obj.device), storage.data_ptr())
        if key in seen_storages:
            return 0
        seen_storages.add(key)
        return storage.nbytes()

    if isinstance(obj, dict):
        return sum(
            tensor_storage_bytes(v) for v in obj.values()
        )

    if isinstance(obj, (list, tuple)):
        return sum(tensor_storage_bytes(v) for v in obj)

    if hasattr(obj, "__dict__"):
        return tensor_storage_bytes(vars(obj))

    return 0

state_bytes = tensor_storage_bytes(optimizer.state)

result = {
    "method": method,
    "parameters": sum(p.numel() for p in model.parameters()),
    "precision": "FP32",
    "batch_size": batch_size,
    "sequence_length": 256,
    "steps": steps,
    "elapsed_seconds": elapsed,
    "tokens_per_second": steps * batch_size * 256 / elapsed,
    "peak_allocated_gib": (
        torch.cuda.max_memory_allocated() / 1024**3
    ),
    "optimizer_tensor_storage_gib_including_projection": (
        state_bytes / 1024**3
    ),
    "initial_loss": losses[0],
    "final_training_batch_loss": losses[-1],
    "first_step_seconds": step_times[0],
    "step_201_seconds": step_times[200],
    "projected_weight_tensors": (
        len(projected_names) if method == "galore" else 0
    ),
}

with open(
    os.path.join(work_dir, f"benchmark_{method}.json"), "w"
) as f:
    json.dump(result, f, indent=2)

print(json.dumps(result, indent=2), flush=True)
'''

script = WORK_DIR / "benchmark.py"
script.write_text(benchmark_code)

jobs = []
for method, gpu in [("adamw", "0"), ("galore", "1")]:
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = gpu
    env["OMP_NUM_THREADS"] = "4"

    log_path = WORK_DIR / f"benchmark_{method}.log"
    log_handle = open(log_path, "w")

    process = subprocess.Popen(
        [sys.executable, "-u", str(script), method, str(WORK_DIR)],
        env=env,
        stdout=log_handle,
        stderr=subprocess.STDOUT,
    )
    jobs.append((method, process, log_handle, log_path))
    print(f"Launched {method} on physical GPU {gpu}", flush=True)

for method, process, log_handle, log_path in jobs:
    exit_code = process.wait()
    log_handle.close()

    if exit_code != 0:
        print(log_path.read_text()[-6000:])
        raise RuntimeError(f"{method} benchmark failed.")

    result_path = WORK_DIR / f"benchmark_{method}.json"
    print(f"\n{method.upper()} RESULTS")
    print(result_path.read_text())

In [ ]:
# STEP 4: Short learning-rate search with held-out evaluation
# Requires benchmark_code from Cell 3.

import os
import sys
import json
import subprocess
from pathlib import Path

WORK_DIR = Path("/kaggle/working/galore_reproduction")

pilot_code = benchmark_code

# One pass through the 4096 training blocks:
# 256 steps × 16 sequences × 256 tokens = 1,048,576 tokens.
pilot_code = pilot_code.replace("steps = 220", "steps = 256")
pilot_code = pilot_code.replace(
    "lr=1e-4,",
    'lr=float(os.environ["PILOT_LR"]),'
)

evaluation_code = r'''
# Token-weighted held-out next-token loss.
model.eval()
validation = data["validation"]
weighted_loss = 0.0
target_count = 0

torch.cuda.synchronize()
evaluation_start = time.perf_counter()

with torch.no_grad():
    for offset in range(0, len(validation), batch_size):
        x = validation[offset:offset + batch_size].to(device)
        output = model(input_ids=x, labels=x)

        # Causal LM predicts positions 1 through sequence_length - 1.
        count = x.shape[0] * (x.shape[1] - 1)
        weighted_loss += output.loss.item() * count
        target_count += count

torch.cuda.synchronize()
validation_loss = weighted_loss / target_count
validation_perplexity = float(torch.exp(
    torch.tensor(validation_loss, dtype=torch.float64)
))
evaluation_seconds = time.perf_counter() - evaluation_start
'''

# Evaluate after the training timer finishes.
pilot_code = pilot_code.replace(
    "# Include tensors stored inside GaLore projector objects.",
    evaluation_code
    + "\n# Include tensors stored inside GaLore projector objects."
)

pilot_code = pilot_code.replace(
    'with open(\n    os.path.join(work_dir, f"benchmark_{method}.json"), "w"\n)',
    '''result.update({
    "learning_rate": float(os.environ["PILOT_LR"]),
    "validation_loss": validation_loss,
    "validation_perplexity": validation_perplexity,
    "evaluation_seconds": evaluation_seconds,
})

with open(
    os.path.join(
        work_dir,
        f"pilot_{method}_{os.environ['PILOT_LR']}.json"
    ), "w"
)'''
)

assert 'lr=float(os.environ["PILOT_LR"])' in pilot_code
assert '"validation_perplexity": validation_perplexity' in pilot_code
assert "steps = 256" in pilot_code

pilot_script = WORK_DIR / "pilot.py"
pilot_script.write_text(pilot_code)

# Different optimizer rates are intentional.
# GaLore scales projected updates by 0.25.
rate_pairs = [
    ("0.0001", "0.001"),
    ("0.0005", "0.005"),
    ("0.001",  "0.01"),
]

all_results = []

for round_number, (adam_lr, galore_lr) in enumerate(rate_pairs, 1):
    print(f"\nStarting pilot round {round_number}/3", flush=True)
    jobs = []

    for method, gpu, lr in [
        ("adamw", "0", adam_lr),
        ("galore", "1", galore_lr),
    ]:
        env = os.environ.copy()
        env["CUDA_VISIBLE_DEVICES"] = gpu
        env["OMP_NUM_THREADS"] = "4"
        env["PILOT_LR"] = lr

        log_path = WORK_DIR / f"pilot_{method}_{lr}.log"
        handle = open(log_path, "w")

        process = subprocess.Popen(
            [
                sys.executable, "-u", str(pilot_script),
                method, str(WORK_DIR),
            ],
            env=env,
            stdout=handle,
            stderr=subprocess.STDOUT,
        )
        jobs.append((method, lr, process, handle, log_path))

    for method, lr, process, handle, log_path in jobs:
        exit_code = process.wait()
        handle.close()

        if exit_code != 0:
            print(log_path.read_text()[-5000:])
            raise RuntimeError(f"Pilot failed: {method}, LR={lr}")

        result_path = WORK_DIR / f"pilot_{method}_{lr}.json"
        result = json.loads(result_path.read_text())
        all_results.append(result)

        print(
            f"{method:7s} | LR={lr:7s} | "
            f"validation loss={result['validation_loss']:.4f} | "
            f"perplexity={result['validation_perplexity']:.2f}",
            flush=True,
        )

summary_path = WORK_DIR / "pilot_summary.json"
summary_path.write_text(json.dumps(all_results, indent=2))

print("\nBest short-pilot configuration per optimizer:")
for method in ["adamw", "galore"]:
    candidates = [
        r for r in all_results if r["method"] == method
    ]
    best = min(candidates, key=lambda r: r["validation_loss"])
    print(
        f"{method}: LR={best['learning_rate']}, "
        f"validation perplexity={best['validation_perplexity']:.2f}"
    )

print("\nSaved:", summary_path)

In [ ]:
# STEP 5: Prepare the main training and final evaluation data
import json
import time
import torch
from pathlib import Path

WORK_DIR = Path("/kaggle/working/galore_reproduction")

# 24,576 steps × 16 sequences × 256 tokens.
MAIN_TRAIN_BLOCKS = 393216

main_train = prepare_packed_blocks("train", MAIN_TRAIN_BLOCKS)

generator = torch.Generator().manual_seed(42)
main_train = main_train[
    torch.randperm(len(main_train), generator=generator)
]

# First 512 blocks were used for pilot tuning.
# Use the following 512 blocks for the main evaluation.
validation_pool = prepare_packed_blocks("validation", 1024)
main_validation = validation_pool[512:].clone()

assert main_train.max().item() < 32000
assert main_validation.max().item() < 32000

torch.save(
    {
        "train": main_train,
        "validation": main_validation,
        "sequence_length": 256,
        "tokenizer": tokenizer.name_or_path,
        "seed": 42,
    },
    WORK_DIR / "main_data.pt",
)

print("Training shape:", tuple(main_train.shape))
print("Training token positions:", main_train.numel())
print("Evaluation shape:", tuple(main_validation.shape))
print("Saved:", WORK_DIR / "main_data.pt")

# Release the large notebook-side tensors.
del main_train, main_validation, validation_pool

In [ ]:
# STEP 6: Launch the main comparison in the background
import os
import sys
import subprocess
from pathlib import Path

WORK_DIR = Path("/kaggle/working/galore_reproduction")

main_code = benchmark_code
main_code = main_code.replace(
    '"benchmark_data.pt"', '"main_data.pt"'
)
main_code = main_code.replace(
    "steps = 220", "steps = len(train) // batch_size"
)
main_code = main_code.replace(
    "lr=1e-4,",
    'lr=float(os.environ["MAIN_LR"]),'
)

# Insert evaluation and schedule helpers before training.
helpers = r'''
import math

base_lr = float(os.environ["MAIN_LR"])
warmup_steps = int(0.1 * steps)
validation_history = []
evaluation_total_seconds = 0.0

def set_learning_rate(step):
    if step < warmup_steps:
        factor = (step + 1) / warmup_steps
    else:
        progress = (step - warmup_steps) / max(
            1, steps - warmup_steps - 1
        )
        factor = 0.1 + 0.9 * 0.5 * (
            1.0 + math.cos(math.pi * progress)
        )

    for group in optimizer.param_groups:
        group["lr"] = base_lr * factor

def evaluate():
    model.eval()
    total_loss = 0.0
    total_targets = 0
    validation = data["validation"]

    torch.cuda.synchronize()
    eval_start = time.perf_counter()

    with torch.no_grad():
        for offset in range(0, len(validation), batch_size):
            x_val = validation[
                offset:offset + batch_size
            ].to(device)

            out = model(input_ids=x_val, labels=x_val)
            count = x_val.shape[0] * (x_val.shape[1] - 1)
            total_loss += out.loss.item() * count
            total_targets += count

    torch.cuda.synchronize()
    duration = time.perf_counter() - eval_start
    model.train()

    mean_loss = total_loss / total_targets
    return mean_loss, math.exp(mean_loss), duration

def save_progress(step):
    global evaluation_total_seconds

    val_loss, perplexity, duration = evaluate()
    evaluation_total_seconds += duration

    record = {
        "step": step,
        "token_positions": step * batch_size * 256,
        "validation_loss": val_loss,
        "validation_perplexity": perplexity,
        "elapsed_seconds": time.perf_counter() - start,
    }
    validation_history.append(record)

    progress = {
        "method": method,
        "base_learning_rate": base_lr,
        "planned_steps": steps,
        "completed_steps": step,
        "validation_history": validation_history,
        "recent_training_loss": (
            losses[-1] if losses else None
        ),
    }

    destination = os.path.join(
        work_dir, f"progress_{method}.json"
    )
    temporary = destination + ".tmp"
    with open(temporary, "w") as f:
        json.dump(progress, f, indent=2)
    os.replace(temporary, destination)

    print(
        f"{method}: step={step}/{steps}, "
        f"validation PPL={perplexity:.2f}",
        flush=True,
    )
'''

main_code = main_code.replace(
    "torch.cuda.empty_cache()\ntorch.cuda.reset_peak_memory_stats()",
    helpers
    + "\ntorch.cuda.empty_cache()\n"
    + "torch.cuda.reset_peak_memory_stats()"
)

main_code = main_code.replace(
    "for step in range(steps):",
    "save_progress(0)\n\nfor step in range(steps):\n"
    "    set_learning_rate(step)"
)

main_code = main_code.replace(
    "    if (step + 1) % 50 == 0:",
    "    if (step + 1) % 1000 == 0 or step + 1 == steps:\n"
    "        save_progress(step + 1)\n\n"
    "    if (step + 1) % 1000 == 0:"
)

# Capture final metadata and save model weights.
main_code = main_code.replace(
    'with open(\n    os.path.join(work_dir, f"benchmark_{method}.json"), "w"\n)',
    '''result.update({
    "base_learning_rate": base_lr,
    "warmup_steps": warmup_steps,
    "schedule": "10% warmup, cosine decay to 10% of base LR",
    "validation_history": validation_history,
    "final_validation_loss": validation_history[-1]["validation_loss"],
    "final_validation_perplexity": validation_history[-1]["validation_perplexity"],
    "evaluation_total_seconds": evaluation_total_seconds,
    "training_step_tokens_per_second": (
        steps * batch_size * 256 / sum(step_times)
    ),
    "timing_note": (
        "elapsed_seconds includes periodic evaluation; "
        "training_step_tokens_per_second excludes evaluation"
    ),
    "gaLore_rank": 128 if method == "galore" else None,
    "projection_refresh_steps": 200 if method == "galore" else None,
    "gaLore_scale": 0.25 if method == "galore" else None,
})

with open(
    os.path.join(work_dir, f"main_{method}.json"), "w"
)'''
)

main_code += r'''

# Save portable CPU weights for later evaluation.
torch.save(
    {
        "model_state_dict": {
            name: tensor.detach().cpu()
            for name, tensor in model.state_dict().items()
        },
        "config": config.to_dict(),
        "result": result,
    },
    os.path.join(work_dir, f"final_{method}.pt"),
)
print(f"{method}: final checkpoint saved", flush=True)
'''

assert 'float(os.environ["MAIN_LR"])' in main_code
assert "save_progress(0)" in main_code
assert '"main_data.pt"' in main_code
assert 'f"main_{method}.json"' in main_code

main_script = WORK_DIR / "main_train.py"
main_script.write_text(main_code)

main_jobs = {}

for method, gpu, lr in [
    ("adamw", "0", "0.0005"),
    ("galore", "1", "0.001"),
]:
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = gpu
    env["OMP_NUM_THREADS"] = "4"
    env["MAIN_LR"] = lr

    log_path = WORK_DIR / f"main_{method}.log"
    with open(log_path, "w") as handle:
        process = subprocess.Popen(
            [
                sys.executable, "-u", str(main_script),
                method, str(WORK_DIR),
            ],
            env=env,
            stdout=handle,
            stderr=subprocess.STDOUT,
        )

    main_jobs[method] = process
    print(
        f"{method}: GPU={gpu}, LR={lr}, PID={process.pid}",
        flush=True,
    )

print("\nTraining launched. Keep the Kaggle session running.")

In [ ]:
# STEP 7: Check status without waiting for training to finish
import json

for method, process in main_jobs.items():
    print(f"\n{method.upper()} | process status: {process.poll()}")
    # None means still running; 0 means completed successfully.

    path = WORK_DIR / f"progress_{method}.json"
    if path.exists():
        progress = json.loads(path.read_text())
        latest = progress["validation_history"][-1]
        print(
            f"Step: {progress['completed_steps']:,}/"
            f"{progress['planned_steps']:,}"
        )
        print(
            f"Validation perplexity: "
            f"{latest['validation_perplexity']:.2f}"
        )

    log_path = WORK_DIR / f"main_{method}.log"
    if log_path.exists():
        print(log_path.read_text()[-1200:])

In [ ]:
import subprocess
from pathlib import Path

work = Path("/kaggle/working/galore_reproduction")

for method, process in main_jobs.items():
    print(f"\n{method.upper()}")
    print("Process status:", process.poll())
    # None = running; 0 = success; another value = failure.

    log = work / f"main_{method}.log"
    print(log.read_text()[-2500:] if log.exists() else "No log yet")

    progress = work / f"progress_{method}.json"
    if progress.exists():
        print("Progress:", progress.read_text()[-1500:])

print("\nGPU STATUS")
print(subprocess.run(
    ["nvidia-smi"],
    capture_output=True,
    text=True,
).stdout)

In [ ]:
from pathlib import Path
import json

work = Path("/kaggle/working/galore_reproduction")
print("Experiment folder exists:", work.exists())

if work.exists():
    print("\nFILES")
    for path in sorted(work.iterdir()):
        print(path.name, f"{path.stat().st_size / 1024:.1f} KiB")

    for method in ["adamw", "galore"]:
        print(f"\n{method.upper()}")

        result = work / f"main_{method}.json"
        progress = work / f"progress_{method}.json"
        checkpoint = work / f"final_{method}.pt"
        log = work / f"main_{method}.log"

        print("Final result exists:", result.exists())
        print("Model checkpoint exists:", checkpoint.exists())

        if result.exists():
            data = json.loads(result.read_text())
            for key in [
                "steps",
                "final_validation_perplexity",
                "peak_allocated_gib",
                "optimizer_tensor_storage_gib_including_projection",
                "elapsed_seconds",
            ]:
                print(key, ":", data.get(key))

        elif progress.exists():
            data = json.loads(progress.read_text())
            print("Last recorded step:", data["completed_steps"])
            print("Latest validation:",
                  data["validation_history"][-1])

        if log.exists():
            print("LOG END:\n", log.read_text()[-1500:])
else:
    print("\nCheck Kaggle's saved notebook versions and Output files.")